# Aarohan — Stage 2: Full-Dataset Exploratory Data Analysis

**Project:** Aarohan (Intelligent Demand Planning for Perishable Retail)  
**Dataset:** FreshRetailNet-50K (Dingdong Inc.)  
**Partitions:** `data/raw/train.parquet` (4.5M rows) & `data/raw/eval.parquet` (350k rows)  
**Execution Constraints:** DuckDB memory limit 512 MB, 2 threads, streaming execution.  

This notebook provides a reproducible, end-to-end exploratory analysis across all 4.85 million records, calling modular functions from `aarohan.analysis`.

In [1]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))


from aarohan.analysis.diagnostics import (
    calculate_stockout_severity_summary,
    calculate_zero_sales_diagnostics,
    compare_train_eval_populations,
)
from aarohan.analysis.profiling import (
    calculate_daily_demand_summary,
    calculate_overall_demand_summary,
    calculate_product_summary,
    calculate_series_distribution_summary,
    calculate_series_profiles,
    calculate_weekday_patterns,
    define_demand_segments,
)
from aarohan.config import load_config
from aarohan.data.load import duckdb_cursor, resolve_dataset_path

data_cfg = load_config("data")
train_path = resolve_dataset_path(data_cfg, "raw_train_path")
eval_path = resolve_dataset_path(data_cfg, "raw_eval_path")
print(f"Project root: {project_root}")
print(f"Train path: {train_path}")
print(f"Eval path: {eval_path}")

Project root: D:\Aarohan
Train path: D:/Aarohan/data/raw/train.parquet
Eval path: D:/Aarohan/data/raw/eval.parquet


## 1. Overall Demand Distribution

We compute observation-level distribution statistics and quantiles across all 4.5 million training observations, as well as daily aggregates.

In [2]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    overall_summary = calculate_overall_demand_summary(con, train_path.as_posix())
    daily_df = calculate_daily_demand_summary(con, train_path.as_posix())

print("--- Observation-Level Demand Distribution (4,500,000 rows) ---")
for k, v in overall_summary.items():
    print(f"  {k}: {v}")

cols = [
    "dt",
    "weekday_name",
    "total_demand",
    "mean_demand",
    "median_demand",
    "zero_sales_pct",
    "mean_daytime_stockout_pct",
]
daily_df[cols].head(7)

--- Observation-Level Demand Distribution (4,500,000 rows) ---
  total_observations: 4500000
  total_demand: 4493660.787
  mean_demand: 0.9986
  median_demand: 0.7
  std_demand: 1.4067
  min_demand: 0.0
  max_demand: 44.9
  p25_demand: 0.4
  p50_demand: 0.7
  p75_demand: 1.1
  p90_demand: 2.0
  p95_demand: 2.9
  p99_demand: 5.8
  zero_sales_count: 200760
  zero_sales_pct: 4.4613


,dt,weekday_name,total_demand,mean_demand,median_demand,zero_sales_pct,mean_daytime_stockout_pct
0,2024-03-28,Thursday,35787.337,0.7157,0.50,6.512,25.3450
1,2024-03-29,Friday,34893.476,0.6979,0.50,7.056,19.5299
2,2024-03-30,Saturday,47995.157,0.9599,0.70,4.724,18.5361
3,2024-03-31,Sunday,49950.534,0.9990,0.70,4.900,22.3371
4,2024-04-01,Monday,35607.989,0.7122,0.50,6.856,19.9234
5,2024-04-02,Tuesday,38055.177,0.7611,0.58,6.382,22.9593
6,2024-04-03,Wednesday,35590.602,0.7118,0.50,5.700,20.6520


## 2. Store-Product Series Heterogeneity & Demand Segmentation

We summarize all 50,000 store-product series across the 90 training dates (100% complete balanced panel), evaluating mean demand, CV (with NaN for zero-mean series), zero-sales pct, and operational segment boundaries.

In [3]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    series_df = calculate_series_profiles(con, train_path.as_posix())
    series_summary_df = calculate_series_distribution_summary(series_df)
    series_df, thresholds_df = define_demand_segments(series_df)

print("--- Series Distribution Summary Moments (50,000 series) ---")
display(series_summary_df)
print("\n--- Operational Segmentation Boundaries ---")
display(thresholds_df)

--- Series Distribution Summary Moments (50,000 series) ---


,metric,valid_series_count,mean,std,min,p10,p25,median,p75,p90,p99,max
0,mean_demand,50000,0.9986,1.1526,0.2220,0.4338,0.4942,0.6578,1.0544,1.8222,4.9478,19.0744
1,median_demand,50000,0.9410,1.2227,0.0000,0.4000,0.5000,0.6000,1.0000,1.7000,4.7500,22.7500
2,std_demand,50000,0.5303,0.6136,0.1018,0.2228,0.2702,0.3684,0.5829,0.9417,2.5860,12.2109
3,cv,50000,0.5665,0.2093,0.1643,0.3741,0.4309,0.5141,0.6414,0.8297,1.3318,3.0078
4,zero_sales_pct,50000,4.4613,7.0650,0.0000,0.0000,0.0000,2.2222,5.5556,12.2222,34.4444,72.2222
5,mean_daytime_stockout_pct,50000,19.8771,8.2140,3.8889,13.0556,14.8611,17.6736,21.9444,28.8889,53.0563,97.0139
6,day_count,50000,90.0000,0.0000,90.0000,90.0000,90.0000,90.0000,90.0000,90.0000,90.0000,90.0000



--- Operational Segmentation Boundaries ---


,segment_dimension,segment_name,metric,lower_bound,upper_bound,series_count,series_pct,mean_demand,mean_daytime_stockout_pct
0,Demand Volume,Low,mean_demand,0.0000,0.5367,16673,33.35,0.4608,19.69
1,Demand Volume,Medium,mean_demand,0.5367,0.8713,16662,33.32,0.6726,19.71
2,Demand Volume,High,mean_demand,0.8713,19.0744,16665,33.33,1.8625,20.23
3,Intermittency,Regular,zero_sales_pct,0.0000,20.0000,47427,94.85,1.0153,19.02
4,Intermittency,Intermittent,zero_sales_pct,20.0000,100.0000,2573,5.15,0.6902,35.70
5,Stockout Exposure,Low,mean_daytime_stockout_pct,0.0000,15.7600,16806,33.61,0.8962,13.68
6,Stockout Exposure,Medium,mean_daytime_stockout_pct,15.7600,20.2100,16675,33.35,0.9650,17.81
7,Stockout Exposure,High,mean_daytime_stockout_pct,20.2100,97.0100,16519,33.04,1.1367,28.27


## 3. Product-Level Distribution & Concentration

Across all 865 products, we examine store penetration, volume concentration, and stockout frequency.

In [4]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    product_df = calculate_product_summary(con, train_path.as_posix())

top_share = (
    product_df.head(len(product_df) // 10)["total_demand"].sum()
    * 100.0
    / overall_summary["total_demand"]
)
print(f"Total unique products: {len(product_df)}")
print(f"Top 10% products share of demand: {top_share:.2f}%")
print("\nTop 5 products by total demand:")
product_df.head(5)

Total unique products: 865
Top 10% products share of demand: 72.30%

Top 5 products by total demand:


,product_id,store_count,observation_count,total_demand,mean_demand,median_demand,std_demand,zero_sales_pct,mean_daytime_stockout_pct
0,267,131,11790,183395.100,15.5551,17.9,9.3545,3.1043,37.0102
1,4,783,70470,183265.300,2.6006,1.7,2.7918,4.1493,18.8571
2,300,412,37080,154505.577,4.1668,3.8,2.3808,2.5566,21.7100
3,117,459,41310,150032.945,3.6319,3.5,1.9122,2.7645,19.1933
4,215,614,55260,144031.118,2.6064,2.5,1.5177,4.8389,23.3471


## 4. Stockout Severity and Availability Diagnostics

We compare observed sales performance and zero-sales frequency across daytime-stockout-free, partially affected, and fully daytime-stockout days, noting that fully 24-hour stockouts form a strict subset of fully daytime stockouts.

In [5]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    severity_df = calculate_stockout_severity_summary(con, train_path.as_posix())
    zero_diag = calculate_zero_sales_diagnostics(con, train_path.as_posix())

print("--- Demand by Stockout Severity Category ---")
display(severity_df)
print("\n--- Zero-Sales Decomposition ---")
for k, v in zero_diag.items():
    print(f"  {k}: {v}")

--- Demand by Stockout Severity Category ---


,stockout_category,category_type,observation_count,observation_pct,mean_sale,median_sale,std_sale,min_sale,max_sale,zero_sales_count,zero_sales_pct,mean_daytime_stockout_hours,mean_24h_stockout_hours
0,Daytime Stockout-Free,Mutually Exclusive Partition,2507994,55.7332,0.9745,0.68,1.2758,0.0,44.9,46944.0,1.8718,0.0000,1.2591
1,Partially Stockout-Affected,Mutually Exclusive Partition,1810215,40.2270,1.1276,0.70,1.5974,0.0,40.4,5771.0,0.3188,6.2992,10.7071
2,Fully Daytime Stockout,Mutually Exclusive Partition,181791,4.0398,0.0460,0.00,0.2013,0.0,15.4,148045.0,81.4369,16.0000,23.8551
3,Fully 24-Hour Stockout,Overlapping Subset of Fully Daytime Stockout,170717,3.7937,0.0369,0.00,0.1948,0.0,15.4,145521.0,85.2411,16.0000,24.0000



--- Zero-Sales Decomposition ---
  total_observations: 4500000
  total_zero_sales: 200760
  total_zero_sales_pct: 4.4613
  in_stock_zero_sales: 46944
  in_stock_zero_sales_pct_of_zeros: 23.38
  in_stock_zero_sales_pct_of_total: 1.0432
  stockout_zero_sales: 153816
  stockout_zero_sales_pct_of_zeros: 76.62
  stockout_zero_sales_pct_of_total: 3.4181
  fully_daytime_stockout_zeros: 148045
  fully_daytime_stockout_nonzeros: 33746
  fully_daytime_stockout_zero_rate_pct: 81.44


## 5. Calendar Patterns: Weekday vs Weekend

Analysis of day-of-week demand and stockout patterns demonstrates substantial weekend lift (+38.6% on Sunday vs Thursday).

In [6]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    weekday_df = calculate_weekday_patterns(con, train_path.as_posix())

weekday_df[
    [
        "weekday_name",
        "is_weekend",
        "date_count",
        "total_demand",
        "mean_demand",
        "median_demand",
        "mean_daytime_stockout_pct",
    ]
]

,weekday_name,is_weekend,date_count,total_demand,mean_demand,median_demand,mean_daytime_stockout_pct
0,Monday,0,13,612677.022,0.9426,0.60,20.1454
1,Tuesday,0,13,625290.406,0.9620,0.66,20.8099
2,Wednesday,0,12,544130.431,0.9069,0.60,19.5325
3,Thursday,0,13,562923.368,0.8660,0.60,18.7119
4,Friday,0,13,602403.771,0.9268,0.60,17.5485
5,Saturday,1,13,765784.513,1.1781,0.80,19.9237
6,Sunday,1,13,780451.276,1.2007,0.80,22.4416


## 6. Training vs Held-Out Evaluation Population Comparison

> **Important**: All evaluation analysis is strictly descriptive and held out. It verifies population alignment and is not used for model tuning.

In [7]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    comp_df = compare_train_eval_populations(
        con, train_path.as_posix(), eval_path.as_posix()
    )

comp_df

,metric,train_partition,eval_partition_held_out,absolute_diff,relative_change,assessment_role
0,Observations,4500000,350000,-4150000,-92.22%,Descriptive Held-Out Assessment: 90 dates train vs 7 dates eval
1,Dates (Days),90,7,-83,-92.22%,Descriptive Held-Out Assessment: 90 days vs 7 days
2,Date Start,2024-03-28,2024-06-26,N/A,N/A,Descriptive Held-Out Assessment: partition start date
3,Date End,2024-06-25,2024-07-02,N/A,N/A,Descriptive Held-Out Assessment: partition end date
4,Unique Stores,898,898,0,0.0%,Identical entity coverage (898 stores)
5,Unique Products,865,865,0,0.0%,Identical entity coverage (865 products)
6,Store-Product Series,50000,50000,0,0.0%,"Identical series coverage (50,000 series)"
7,Mean Daily Demand,0.9986,1.1931,0.1945,19.48%,Descriptive Held-Out: +19.48% higher mean sales in eval
8,Median Daily Demand,0.7,0.8,0.1,14.29%,Descriptive Held-Out: +14.29% higher median sales in eval
9,Std Dev Daily Demand,1.4067,1.9116,0.5049,35.89%,Descriptive Held-Out: +35.89% higher volatility in eval


## 7. Generated Visualizations & Artifacts

All 8 publication-grade vector SVG figures generated by `aarohan.analysis.visualize` are saved in `reports/figures/`:
- `fig01_daily_aggregate_demand.svg`: Daily aggregate demand trend.
- `fig02_series_mean_demand_distribution.svg`: Histogram of series mean demand.
- `fig03_zero_sales_distribution.svg`: Histogram of series zero-sales share.
- `fig04_weekday_demand_pattern.svg`: Weekday demand pattern.
- `fig05_stockout_exposure_distribution.svg`: Daytime stockout hours distribution.
- `fig06_sales_by_stockout_category.svg`: Observed sales across stockout tiers.
- `fig07_demand_segments_stockout_profile.svg`: Demand volume and stockout exposure.
- `fig08_category_heterogeneity.svg`: Product category volume and stockouts.

Complete analytical commentary and architectural guidance for Stage 3 are documented in `reports/eda_findings.md`.